# Pinecone (2026 업데이트판)

Pinecone은 완전 관리형 벡터 데이터베이스입니다. 인덱스 생성부터 임베딩, 하이브리드 검색, 재순위화(reranking)까지 하나의 서비스 안에서 처리할 수 있습니다.

**Pinecone의 장점**
1. 확장성: 대규모 데이터셋에 대해 뛰어난 확장성을 제공합니다.
2. 관리 용이성: 완전 관리형 서비스라 인프라 관리 부담이 적습니다.
3. 실시간 업데이트: 데이터의 실시간 삽입·수정·삭제가 가능합니다.
4. 고가용성: 클라우드 기반으로 높은 가용성과 내구성을 제공합니다.
5. 통합 기능: 임베딩 생성(integrated inference)과 reranking을 서비스 안에서 제공해, 별도 모델 서버 없이 하이브리드 검색과 재순위화를 구성할 수 있습니다.

**Pinecone의 단점**
1. 비용: Chroma나 FAISS에 비해 상대적으로 비용이 높을 수 있습니다.
2. 커스터마이징 제한: 완전 관리형이라 세부 튜닝의 자유도가 낮습니다.
3. 데이터 위치: 클라우드에 데이터를 저장하므로 데이터 주권 문제가 생길 수 있습니다.

소규모 실험이라면 Chroma나 FAISS가, 대규모 운영 환경이라면 Pinecone이 유리합니다.

### 원본 대비 변경 사항

원본은 `langchain_teddynote.community.pinecone` 이라는 **서드파티 헬퍼 모듈**을 중심으로 구성되어 있었습니다. 당시에는 LangChain 공식 통합에 하이브리드 검색·reranking 기능이 없었기 때문입니다. 지금은 `langchain-pinecone` 과 Pinecone SDK가 같은 기능을 공식적으로 제공하므로, 전부 공식 API로 바꿨습니다.

| 항목 | 원본 (`langchain_teddynote`) | 현재 권장 (공식 API) |
|---|---|---|
| LangSmith 설정 | `langchain_teddynote.logging` | 환경 변수 (`LANGSMITH_*`) |
| 인덱스 생성 | `create_index(...)` 헬퍼 | `pinecone.Pinecone.create_index(...)` + `ServerlessSpec` |
| 유료 Pod 인덱스 | `PodSpec(...)` 예제 | **serverless 기본** — pod 기반 인덱스는 2026년 4월 legacy 로 전환 |
| 문서 전처리 | `preprocess_documents(...)` | `Document` 리스트를 그대로 사용 (`metadata` 는 분할 시 지정) |
| 문서 업서트 | `upsert_documents(...)` / `upsert_documents_parallel(...)` | `PineconeVectorStore.from_documents(...)` / `add_documents(...)` |
| 희소 벡터(Sparse) | Kiwi 형태소 분석기 + BM25 인코더를 `pickle` 로 저장 | `PineconeSparseEmbeddings` + `PineconeSparseVectorStore` (서버측 추론, 저장할 파일 없음) |
| 하이브리드 검색 | `PineconeKiwiHybridRetriever` (`alpha` 가중치) | dense/sparse 검색기 2개 + `EnsembleRetriever` (RRF) |
| 한글 불용어 사전 | `langchain_teddynote.korean.stopwords()` | 불필요 (다국어 임베딩 모델이 처리) |
| 네임스페이스 삭제 | `delete_namespace(...)` | `vector_store.delete(delete_all=True, namespace=...)` |
| 필터 삭제 | `delete_by_filter(...)` | `vector_store.delete(filter={...}, namespace=...)` |
| Reranking | `search_kwargs={"rerank": True, ...}` | `PineconeRerank` + `ContextualCompressionRetriever` |
| 샘플 데이터 | `data/*.pdf` (저장소에 포함되지 않음) | 이 장의 `data/*.txt` (1·2번 노트북과 동일) |

**참고**
- [Pinecone LangChain 통합 문서](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone)
- [Pinecone 인덱스 생성 가이드](https://docs.pinecone.io/guides/index-data/create-an-index)
- [Pinecone Rerank 가이드](https://docs.pinecone.io/guides/inference/rerank)

In [ ]:
%pip install -qU langchain-pinecone langchain-classic langchain-openai langchain-text-splitters python-dotenv

## API 키 발급

- [Pinecone 콘솔](https://app.pinecone.io/) 에 접속합니다.
- 프로필 → Account → Projects → API keys 에서 키를 발급합니다.

`.env` 파일에 아래와 같이 추가합니다.

```
PINECONE_API_KEY="YOUR_PINECONE_API_KEY"
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
```

## 환경 설정

**변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()` 는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것입니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH09-VectorStore")

assert os.environ.get("PINECONE_API_KEY"), "PINECONE_API_KEY 를 .env 에 설정하세요."

## 데이터 전처리

**변경점**
- 원본은 `data/*.pdf` 를 읽었지만 이 저장소에는 해당 PDF가 포함되어 있지 않습니다. 1·2번 노트북과 같은 `data/*.txt` 를 사용해 장 전체의 데이터를 통일했습니다. PDF를 쓰고 싶다면 `langchain-pymupdf` 같은 전용 로더 패키지로 읽은 뒤 아래와 동일하게 분할하면 됩니다.
- 원본의 `preprocess_documents(split_docs, metadata_keys=[...], min_length=5, use_basename=True)` 는 ① 필요한 메타데이터만 추리고 ② 짧은 청크를 버리고 ③ 파일 경로를 파일명으로 줄이는 세 가지 일을 했습니다. 이 셋은 분할 단계에서 메타데이터를 직접 지정하고 리스트 컴프리헨션으로 거르면 되므로, 별도 헬퍼가 필요 없습니다.
- `PineconeVectorStore` 는 `Document` 리스트를 그대로 받으므로 `contents` / `metadatas` 로 분리할 필요가 없습니다.

In [ ]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

MIN_LENGTH = 5  # 이보다 짧은 청크는 검색에 도움이 되지 않으므로 제외


def load_and_split(path: str):
    # 파일을 읽어 청크로 나누고, 파일명(basename)을 source 메타데이터로 붙입니다.
    file_path = Path(path)
    raw_text = file_path.read_text(encoding="utf-8")
    docs = text_splitter.create_documents(
        [raw_text], metadatas=[{"source": file_path.name}]
    )
    return [doc for doc in docs if len(doc.page_content) >= MIN_LENGTH]


split_docs = []
for path in sorted(Path("data").glob("*.txt")):
    split_docs.extend(load_and_split(str(path)))

len(split_docs)

In [ ]:
# 첫 번째 청크의 내용과 메타데이터 확인
print(split_docs[0].page_content)
print(split_docs[0].metadata)

## 인덱스 생성

**변경점**
- 원본의 `create_index(...)` 헬퍼는 Pinecone SDK의 `create_index` 를 감싼 것이었습니다. 지금은 SDK를 직접 호출합니다.
- **`PodSpec` 은 쓰지 않습니다.** Pinecone은 2026년 4월부터 serverless 를 기본으로 삼았고, pod 기반 인덱스는 legacy 상태입니다. 신규 프로젝트는 `ServerlessSpec` 으로 시작하세요.
- 원본은 하이브리드 검색을 염두에 두고 dense 인덱스의 `metric` 을 `dotproduct` 로 지정했습니다. 지금은 dense 인덱스와 sparse 인덱스를 **각각** 만들기 때문에, dense 쪽은 `cosine` 을 쓰면 됩니다.

**주요 매개변수**
- `name`: 인덱스 이름
- `dimension`: 임베딩 차원. 모델과 반드시 일치해야 합니다 (`text-embedding-3-small`: 1536, `text-embedding-3-large`: 3072, `multilingual-e5-large`: 1024)
- `metric`: `cosine`, `dotproduct`, `euclidean`
- `vector_type`: `dense`(기본) 또는 `sparse`
- `spec`: `ServerlessSpec(cloud=..., region=...)`
- `deletion_protection`: `"enabled"` 로 두면 실수로 인덱스를 지우는 것을 막아 줍니다

In [ ]:
import time

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "langchain-kr-dense"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        vector_type="dense",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

pc.describe_index(INDEX_NAME)

## 문서 업서트 (Upsert)

**변경점**: 원본의 `upsert_documents(...)` / `upsert_documents_parallel(...)` 헬퍼는 `PineconeVectorStore` 가 대신합니다. 배치 처리와 병렬 요청(`async_req=True`)이 이미 내장되어 있어 별도 코드가 필요 없습니다.

`namespace` 는 하나의 인덱스를 논리적으로 나누는 단위입니다. 같은 인덱스 안에서 프로젝트·테넌트·버전을 분리할 때 사용합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
NAMESPACE = "langchain-kr-01"

vector_store = PineconeVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    batch_size=64,  # 한 번에 업서트할 벡터 수
)

업서트는 비동기로 색인되므로, 바로 검색하면 결과가 비어 있을 수 있습니다. 통계로 반영 여부를 확인합니다.

In [ ]:
index = pc.Index(INDEX_NAME)

# 네임스페이스별 벡터 수와 차원 확인
index.describe_index_stats()

이미 만들어 둔 인덱스에 연결할 때는 `from_existing_index` 를 사용합니다. 업서트 없이 검색만 할 때 쓰는 방법입니다.

In [ ]:
vector_store = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)

## 유사도 검색

`similarity_search(query, k=4, filter=None, namespace=None)` 로 검색합니다. 점수가 필요하면 `similarity_search_with_score` 를 사용하며, `metric="cosine"` 인덱스에서는 값이 클수록 유사합니다.

In [ ]:
results = vector_store.similarity_search("TF IDF 에 대하여 알려줘", k=3)

for doc in results:
    print(doc.page_content[:100])
    print(doc.metadata)
    print("-" * 40)

In [ ]:
for doc, score in vector_store.similarity_search_with_score(
    "TF IDF 에 대하여 알려줘", k=3
):
    print(f"[score={score:.4f}] {doc.page_content[:80]}")

### 메타데이터 필터링

Pinecone은 MongoDB 스타일의 필터 문법을 지원합니다.

- 비교: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`
- 포함: `$in`, `$nin`
- 논리: `$and`, `$or`
- 존재: `$exists`

**변경점**: 원본은 `pinecone_retriever.invoke(query, search_kwargs={"filter": ...})` 라는 서드파티 전용 형태를 썼습니다. 공식 API에서는 `similarity_search(..., filter=...)` 또는 검색기 생성 시 `search_kwargs={"filter": ...}` 로 넘깁니다.

In [ ]:
vector_store.similarity_search(
    "ESG 에 대하여 알려줘",
    k=2,
    filter={"source": {"$eq": "finance-keywords.txt"}},
)

In [ ]:
# 여러 source 중 하나에 속하는 문서만
vector_store.similarity_search(
    "Word2Vec 에 대하여 알려줘",
    k=2,
    filter={"source": {"$in": ["nlp-keywords.txt", "finance-keywords.txt"]}},
)

### MMR 검색

`max_marginal_relevance_search` 는 유사도가 높으면서 서로 겹치지 않는 문서를 고릅니다.

In [ ]:
vector_store.max_marginal_relevance_search(
    "Word2Vec 에 대하여 알려줘", k=3, fetch_k=20, lambda_mult=0.5
)

## 문서 추가 / 삭제

`add_documents(documents, ids=...)` 로 문서를 추가합니다. 같은 ID로 다시 넣으면 덮어쓰기(upsert)됩니다.

In [ ]:
from langchain_core.documents import Document

vector_store.add_documents(
    [
        Document(
            page_content="벡터 데이터베이스는 임베딩을 저장하고 검색하는 데이터베이스입니다.",
            metadata={"source": "mydata.txt"},
        )
    ],
    ids=["new_doc1"],
)

In [ ]:
# ID 로 조회 (표준 VectorStore 인터페이스)
vector_store.get_by_ids(["new_doc1"])

**변경점**: 원본의 `delete_namespace(...)`, `delete_by_filter(...)` 헬퍼는 `vector_store.delete(...)` 하나로 통합됩니다.

- `delete(ids=[...])`: ID 목록으로 삭제
- `delete(filter={...})`: 메타데이터 조건으로 삭제 (**유료 플랜 전용**)
- `delete(delete_all=True, namespace=...)`: 네임스페이스 전체 삭제

In [ ]:
# ID 로 삭제
vector_store.delete(ids=["new_doc1"])

index.describe_index_stats()

## 검색기(Retriever)로 변환

`as_retriever()` 로 체인에 바로 연결할 수 있는 Runnable 을 만듭니다.

In [ ]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

dense_retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# MMR + 메타데이터 필터
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 20,
        "lambda_mult": 0.5,
        "filter": {"source": {"$eq": "finance-keywords.txt"}},
    },
)

mmr_retriever.invoke("ESG 에 대하여 알려줘")

---

## 하이브리드 검색 (Dense + Sparse)

원본은 한국어 키워드 검색을 위해 **Kiwi 형태소 분석기 + BM25 인코더**를 직접 학습시키고 `sparse_encoder.pkl` 로 저장한 뒤, `PineconeKiwiHybridRetriever` 가 `alpha` 가중치로 dense/sparse 점수를 섞었습니다.

지금은 Pinecone이 희소 임베딩을 **서버에서** 만들어 주므로 인코더를 학습하거나 파일로 저장할 일이 없습니다.

**변경점 정리**

| 원본 | 현재 |
|---|---|
| `create_sparse_encoder(stopwords(), mode="kiwi")` | `PineconeSparseEmbeddings(model="pinecone-sparse-english-v0")` |
| `fit_sparse_encoder(..., save_path="./sparse_encoder.pkl")` | 학습·저장 불필요 |
| `load_sparse_encoder("./sparse_encoder.pkl")` | 불필요 |
| 하나의 `dotproduct` 인덱스에 dense+sparse 동시 저장 | dense 인덱스와 sparse 인덱스를 분리 |
| `PineconeKiwiHybridRetriever(alpha=0.5)` | `EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])` |

> ### ⚠️ 한국어 사용 시 주의
>
> Pinecone이 제공하는 희소 임베딩 모델 `pinecone-sparse-english-v0` 는 **영어 전용**입니다. 한국어 문서에는 어휘 매칭 품질을 기대하기 어렵습니다.
>
> 한국어에서는 다음 중 하나를 선택하세요.
> 1. **다국어 dense 임베딩만 사용** — `PineconeEmbeddings(model="multilingual-e5-large")` 는 한국어를 포함한 다국어를 지원하며, 아래 "통합 추론" 절에서 다룹니다. 대부분의 한국어 RAG에서는 이것으로 충분합니다.
> 2. **어휘 검색이 꼭 필요하면 BM25 를 직접 운영** — Kiwi 형태소 분석기로 토크나이징한 BM25 검색기를 로컬에 두고, `EnsembleRetriever` 로 Pinecone dense 검색과 결합합니다. 이 방식은 10장 [`10-Kiwi-BM25Retriever`](../10-Retriever/10-Kiwi-BM25Retriever.ipynb) 와 [`03-EnsembleRetriever`](../10-Retriever/03-EnsembleRetriever.ipynb) 에서 자세히 다룹니다.
>
> 아래 예제는 공식 sparse API의 사용법을 보여 주기 위한 것으로, 영어 문서를 대상으로 합니다.

In [ ]:
from langchain_pinecone import PineconeSparseEmbeddings, PineconeSparseVectorStore

SPARSE_INDEX_NAME = "langchain-kr-sparse"

# 희소 벡터 인덱스는 dimension 을 지정하지 않고 metric 을 dotproduct 로 둡니다.
if not pc.has_index(SPARSE_INDEX_NAME):
    pc.create_index(
        name=SPARSE_INDEX_NAME,
        metric="dotproduct",
        vector_type="sparse",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(SPARSE_INDEX_NAME).status["ready"]:
        time.sleep(1)

sparse_embeddings = PineconeSparseEmbeddings(model="pinecone-sparse-english-v0")

In [ ]:
# 영어 예시 문서 (sparse 모델이 영어 전용이므로)
english_docs = [
    Document(
        page_content="TF-IDF weights a term by how often it appears in a document "
        "and how rare it is across the whole corpus.",
        metadata={"source": "nlp-en.txt"},
    ),
    Document(
        page_content="Word2Vec learns dense word embeddings by predicting "
        "surrounding context words.",
        metadata={"source": "nlp-en.txt"},
    ),
    Document(
        page_content="ESG investing evaluates companies on environmental, social, "
        "and governance criteria.",
        metadata={"source": "finance-en.txt"},
    ),
]

sparse_store = PineconeSparseVectorStore.from_documents(
    documents=english_docs,
    embedding=sparse_embeddings,
    index_name=SPARSE_INDEX_NAME,
    namespace=NAMESPACE,
)

sparse_store.similarity_search("What is TF-IDF?", k=2)

### 두 검색기 결합 (`EnsembleRetriever`)

원본의 `alpha` 파라미터(0: sparse만, 1: dense만)는 `EnsembleRetriever` 의 `weights` 로 대응됩니다. `EnsembleRetriever` 는 RRF(Reciprocal Rank Fusion)로 두 결과의 **순위**를 합치므로, 점수 스케일이 다른 검색기끼리도 안전하게 섞입니다.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

# 같은 영어 문서를 dense 인덱스에도 넣어 두고 비교합니다.
dense_en_store = PineconeVectorStore.from_documents(
    documents=english_docs,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace="en-demo",
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        dense_en_store.as_retriever(search_kwargs={"k": 3}),
        sparse_store.as_retriever(search_kwargs={"k": 3}),
    ],
    weights=[0.5, 0.5],  # 원본의 alpha=0.5 에 해당
)

hybrid_retriever.invoke("What is TF-IDF?")

---

## Pinecone 통합 추론 (Integrated Inference)

Pinecone은 임베딩 모델도 함께 제공합니다. OpenAI 키 없이 Pinecone API 키 하나로 임베딩과 검색을 모두 처리할 수 있고, **`multilingual-e5-large` 는 한국어를 지원**하므로 앞에서 언급한 한국어 하이브리드 문제의 현실적인 대안이 됩니다.

지원 모델(2026년 9월 기준)

| 모델 | 종류 | 차원 | 비고 |
|---|---|---|---|
| `multilingual-e5-large` | dense | 1024 | 다국어(한국어 포함) |
| `llama-text-embed-v2` | dense | 1024~2048 | 차원 선택 가능 |
| `pinecone-sparse-english-v0` | sparse | – | 영어 전용 어휘 기반 |

사용 가능한 모델은 `PineconeEmbeddings(...).list_supported_models()` 로 확인할 수 있습니다.

In [ ]:
from langchain_pinecone import PineconeEmbeddings

MULTILINGUAL_INDEX = "langchain-kr-multilingual"

pinecone_embeddings = PineconeEmbeddings(model="multilingual-e5-large")

if not pc.has_index(MULTILINGUAL_INDEX):
    pc.create_index(
        name=MULTILINGUAL_INDEX,
        dimension=1024,  # multilingual-e5-large 의 차원
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(MULTILINGUAL_INDEX).status["ready"]:
        time.sleep(1)

multilingual_store = PineconeVectorStore.from_documents(
    documents=split_docs,
    embedding=pinecone_embeddings,
    index_name=MULTILINGUAL_INDEX,
    namespace=NAMESPACE,
    batch_size=64,
)

In [ ]:
# 한국어 쿼리로 검색
multilingual_store.similarity_search("ESG 에 대하여 알려줘", k=3)

---

## Reranking

**변경점**: 원본은 `search_kwargs={"rerank": True, "rerank_model": ..., "top_n": ...}` 라는 서드파티 전용 옵션을 썼고, "pinecone 라이브러리 의존성 문제로 동작하지 않을 수 있다"는 단서가 붙어 있었습니다.

지금은 `PineconeRerank` 가 `langchain-pinecone` 에 정식 포함되어 있고, LangChain의 표준 인터페이스인 `BaseDocumentCompressor` 를 구현합니다. 따라서 `ContextualCompressionRetriever` 로 감싸면 **어떤 검색기든** 그대로 재순위화할 수 있습니다.

**주요 매개변수**
- `model`: 재순위화 모델 (기본값 `bge-reranker-v2-m3`. 다국어를 지원해 한국어에도 쓸 수 있습니다)
- `top_n`: 재순위화 후 남길 문서 수 (기본값 3)
- `rank_fields`: 문서가 딕셔너리일 때 기준으로 삼을 필드

사용 가능한 모델은 `PineconeRerank().list_supported_models()` 로 확인합니다.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_pinecone import PineconeRerank

reranker = PineconeRerank(model="bge-reranker-v2-m3", top_n=3)

# 먼저 넉넉히 가져온 뒤(k=10) reranker 가 상위 3개로 추립니다.
base_retriever = multilingual_store.as_retriever(search_kwargs={"k": 10})

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever,
)

In [ ]:
query = "ESG 경영이 기업 가치에 미치는 영향"

print("[Reranker 미사용]")
for doc in base_retriever.invoke(query)[:3]:
    print("-", doc.page_content[:80])

print()
print("[Reranker 적용]")
for doc in compression_retriever.invoke(query):
    score = doc.metadata.get("relevance_score")
    print(f"- (score={score}) {doc.page_content[:80]}")

---

## 정리

실습이 끝나면 인덱스를 삭제해 비용이 발생하지 않도록 합니다. 네임스페이스만 비우려면 `delete(delete_all=True, namespace=...)` 를 사용하세요.

In [ ]:
# 네임스페이스만 비우기
# vector_store.delete(delete_all=True, namespace=NAMESPACE)

# 인덱스 전체 삭제 (되돌릴 수 없습니다)
for name in [INDEX_NAME, SPARSE_INDEX_NAME, MULTILINGUAL_INDEX]:
    if pc.has_index(name):
        pc.delete_index(name)

pc.list_indexes()